# Chemiscope on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-cosmo/chemiscope/blob/main/python/examples/chemiscope-colab.ipynb)

[chemiscope](https://chemiscope.org) is an interactive explorer for collections of
molecules and materials. This notebook runs entirely in the browser on Google Colab:
it installs chemiscope, loads a ready-made dataset, rebuilds one from a structure
file, and draws molecular dipoles and polarizabilities as shapes.

Run the cells in order (`Shift+Enter`).

## 1. Install

Colab does not ship chemiscope or ASE, so install them first. This takes ~30 s.

In [ ]:
%pip install chemiscope ase

## 2. Explore a ready-made dataset

The quickest way to see chemiscope is to load an existing dataset. Here we download
a small example (dipole and polarizability of 29 molecules) straight from the
chemiscope repository and display it with
[`chemiscope.show_input`](https://chemiscope.org/docs/python/widget.html).

The widget below is fully interactive: click a point on the map to show the
corresponding structure, and vice versa.

In [ ]:
import urllib.request

import chemiscope


url = (
    "https://raw.githubusercontent.com/lab-cosmo/chemiscope/main/"
    "docs/src/datasets/showcase.json.gz"
)
urllib.request.urlretrieve(url, "showcase.json.gz")

chemiscope.show_input("showcase.json.gz")

> **If the widget does not appear** (older Colab runtimes), run
> `from google.colab import output; output.enable_custom_widget_manager()` once,
> then re-run the cell above. Most recent Colab runtimes display it without this.

## 3. Build a dataset from a structure file

Section 2 loaded a finished `.json.gz`; this is how such a dataset is produced from a
structure file. We download a realistic
[extended XYZ](https://wiki.fysik.dtu.dk/ase/ase/io/formatoptions.html#extxyz)
file (the same molecules, with dipole and polarizability stored per structure), read
it with [ASE](https://wiki.fysik.dtu.dk/ase), let `extract_properties` pull the stored
properties, and pick the map axes and color with `quick_settings`.

The same recipe works on **your own** files: replace the download with
`ase.io.read("my_structures.xyz", ":")` (upload a file via the Colab file browser on
the left, or mount Google Drive).

In [ ]:
import ase.io


url = (
    "https://raw.githubusercontent.com/lab-cosmo/chemiscope/main/"
    "python/examples/data/showcase.xyz"
)
urllib.request.urlretrieve(url, "showcase.xyz")

structures = ase.io.read("showcase.xyz", ":")

# extract_properties reads the dipole/polarizability stored in the file. Each is a
# vector/tensor, so chemiscope exposes the components as dipole_ccsd[1..3] and
# ccsd_pol[1..6].
properties = chemiscope.extract_properties(structures, only=["dipole_ccsd", "ccsd_pol"])

chemiscope.show(
    structures=structures,
    properties=properties,
    metadata={"name": "Dipole and polarizability"},
    settings=chemiscope.quick_settings(
        x="ccsd_pol[1]", y="ccsd_pol[2]", map_color="dipole_ccsd[1]"
    ),
)

## 4. Draw the dipole and polarizability as shapes

The dataset stores a dipole **vector** and a polarizability **tensor** for each
molecule, and chemiscope can render both in the 3D viewer:
`ase_vectors_to_arrows` turns the dipole into an arrow, and
`ase_tensors_to_ellipsoids` turns the polarizability tensor into an ellipsoid. We
center each molecule first so these per-structure shapes sit on it, then activate
both through the `shape` setting.

Select different points on the map to compare the dipole and polarizability across
molecules.

In [ ]:
# center each molecule so the per-structure shapes sit on it (a translation does
# not change the dipole or the polarizability)
for structure in structures:
    structure.positions -= structure.get_center_of_mass()

shapes = {
    "dipole": chemiscope.ase_vectors_to_arrows(
        structures, "dipole_ccsd", scale=0.5, radius=0.2
    ),
    "polarizability": chemiscope.ase_tensors_to_ellipsoids(
        structures, "ccsd_pol", force_positive=True, scale=0.2
    ),
}

settings = chemiscope.quick_settings(
    x="ccsd_pol[1]", y="ccsd_pol[2]", map_color="dipole_ccsd[1]"
)
# display both shapes in the structure panel
settings["structure"][0]["shape"] = "dipole,polarizability"

chemiscope.show(
    structures=structures,
    properties=properties,
    metadata={"name": "Dipole and polarizability shapes"},
    shapes=shapes,
    settings=settings,
)

## Next steps

- More examples: <https://chemiscope.org/docs/examples/>
- Python API reference: <https://chemiscope.org/docs/python/>
- Save a dataset to share or open at [chemiscope.org](https://chemiscope.org):
  `chemiscope.write_input("my_dataset.json.gz", structures=..., properties=...)`